<a href="https://colab.research.google.com/github/Aari2066/AI_LAB/blob/main/1WA24CS004_Week6_PropositionalLogic_TruthTableEnumeration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from itertools import product
import re

def tokenize(expr):
    return re.findall(r'<->|->|[()!&|]|[A-Za-z][A-Za-z0-9_]*', expr)


class Parser:
    def __init__(self, expr):
        self.tokens = tokenize(expr)
        self.pos = 0

    def match(self, token):
        if self.pos < len(self.tokens) and self.tokens[self.pos] == token:
            self.pos += 1
            return True
        return False

    def parse(self):
        result = self.iff()
        if self.pos != len(self.tokens):
            raise ValueError("Invalid expression")
        return result

    def iff(self):
        left = self.implies()
        while self.match('<->'):
            right = self.implies()
            left = ('iff', left, right)
        return left

    def implies(self):
        left = self.or_expr()
        if self.match('->'):
            right = self.implies()
            return ('implies', left, right)
        return left

    def or_expr(self):
        left = self.and_expr()
        while self.match('|'):
            right = self.and_expr()
            left = ('or', left, right)
        return left

    def and_expr(self):
        left = self.not_expr()
        while self.match('&'):
            right = self.not_expr()
            left = ('and', left, right)
        return left

    def not_expr(self):
        if self.match('!'):
            return ('not', self.not_expr())

        if self.match('('):
            result = self.iff()
            if not self.match(')'):
                raise ValueError("Missing closing parenthesis")
            return result

        if self.pos >= len(self.tokens):
            raise ValueError("Invalid expression")

        token = self.tokens[self.pos]
        self.pos += 1

        if token in ('&', '|', '->', '<->', ')'):
            raise ValueError("Invalid expression")

        return ('var', token)


def evaluate(tree, model):
    op = tree[0]

    if op == 'var':
        return model[tree[1]]

    if op == 'not':
        return not evaluate(tree[1], model)

    left = evaluate(tree[1], model)
    right = evaluate(tree[2], model)

    if op == 'and':
        return left and right

    if op == 'or':
        return left or right

    if op == 'implies':
        return not left or right

    if op == 'iff':
        return left == right


def get_symbols(tree):
    if tree[0] == 'var':
        return {tree[1]}

    if tree[0] == 'not':
        return get_symbols(tree[1])

    return get_symbols(tree[1]) | get_symbols(tree[2])


def truth_table_entailment(kb, query):
    symbols = sorted(set().union(
        *(get_symbols(sentence) for sentence in kb),
        get_symbols(query)
    ))

    models = []
    entails = True

    print("\nTRUTH TABLE")
    print("-" * 65)

    print(" | ".join(symbols + [f"KB{i+1}" for i in range(len(kb))]
                       + ["KB", "Query"]))

    print("-" * 65)

    for values in product([True, False], repeat=len(symbols)):
        model = dict(zip(symbols, values))

        kb_values = [evaluate(sentence, model) for sentence in kb]
        kb_result = all(kb_values)
        query_result = evaluate(query, model)

        row = ["T" if model[s] else "F" for s in symbols]
        row += ["T" if value else "F" for value in kb_values]
        row += ["T" if kb_result else "F"]
        row += ["T" if query_result else "F"]

        print(" | ".join(row))

        if kb_result:
            models.append(model)

            if not query_result:
                entails = False

    print("\nMODELS OF KB:")

    for model in models:
        print(model)

    print("\nTOTAL MODELS:", len(models))

    print("\nENTAILMENT RESULT:")
    if entails:
        print("KB entails the query.")
    else:
        print("KB does NOT entail the query.")

    return entails


print("PROPOSITIONAL LOGIC - TRUTH TABLE ENUMERATION")

n = int(input("\nEnter number of sentences in KB: "))

kb = []

print("\nEnter KB sentences:")
print("Operators: ! (NOT), & (AND), | (OR), -> (IMPLIES), <-> (BICONDITIONAL)")

for i in range(n):
    expression = input(f"KB Sentence {i+1}: ")
    kb.append(Parser(expression).parse())

m = int(input("\nEnter number of queries: "))

queries = []

for i in range(m):
    expression = input(f"Query {i+1}: ")
    queries.append(Parser(expression).parse())3

for i, query in enumerate(queries):
    print(f"\n========== QUERY {i+1} ==========")
    truth_table_entailment(kb, query)


PROPOSITIONAL LOGIC - TRUTH TABLE ENUMERATION

Enter number of sentences in KB: 30

Enter KB sentences:
Operators: ! (NOT), & (AND), | (OR), -> (IMPLIES), <-> (BICONDITIONAL)
KB Sentence 1: WEE
KB Sentence 2: RRR
KB Sentence 3: EE
KB Sentence 4: 444


ValueError: Invalid expression